In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import shutil
shutil.copy('/content/drive/MyDrive/DOUTORADO/DATASETS/VOOS_2025/VRA_20251.csv', 'VRA_20251.csv')
shutil.copy('/content/drive/MyDrive/DOUTORADO/DATASETS/VOOS_2025/VRA_20252.csv', 'VRA_20252.csv')
shutil.copy('/content/drive/MyDrive/DOUTORADO/DATASETS/VOOS_2025/VRA_20253.csv', 'VRA_20253.csv')
shutil.copy('/content/drive/MyDrive/DOUTORADO/DATASETS/VOOS_2025/VRA_20254.csv', 'VRA_20254.csv')
shutil.copy('/content/drive/MyDrive/DOUTORADO/DATASETS/VOOS_2025/VRA_20255.csv', 'VRA_20255.csv')
shutil.copy('/content/drive/MyDrive/DOUTORADO/DATASETS/VOOS_2025/VRA_20256.csv', 'VRA_20256.csv')
shutil.copy('/content/drive/MyDrive/DOUTORADO/DATASETS/VOOS_2025/VRA_20257.csv', 'VRA_20257.csv')
shutil.copy('/content/drive/MyDrive/DOUTORADO/DATASETS/VOOS_2025/VRA_20258.csv', 'VRA_20258.csv')
shutil.copy('/content/drive/MyDrive/DOUTORADO/DATASETS/VOOS_2025/VRA_20259.csv', 'VRA_20259.csv')
shutil.copy('/content/drive/MyDrive/DOUTORADO/DATASETS/VOOS_2025/VRA_202510.csv', 'VRA_202510.csv')
shutil.copy('/content/drive/MyDrive/DOUTORADO/DATASETS/VOOS_2025/VRA_202511.csv', 'VRA_202511.csv')

'VRA_202511.csv'

In [ ]:
import pandas as pd
import glob

paths = glob.glob('/content/VRA_2025*.csv')

df = pd.concat(
    (pd.read_csv(p, sep=';', low_memory=False) for p in paths),
    ignore_index=True
)


In [ ]:
df.info()
df.head()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 917919 entries, 0 to 917918
Data columns (total 12 columns):
 #   Column                   Non-Null Count   Dtype  
---  ------                   --------------   -----  
 0   ICAO Empresa Aérea       917919 non-null  object 
 1   Número Voo               917919 non-null  object 
 2   Código Autorização (DI)  917919 non-null  object 
 3   Código Tipo Linha        917446 non-null  object 
 4   ICAO Aeródromo Origem    917919 non-null  object 
 5   ICAO Aeródromo Destino   917919 non-null  object 
 6   Partida Prevista         892120 non-null  object 
 7   Partida Real             892717 non-null  object 
 8   Chegada Prevista         892120 non-null  object 
 9   Chegada Real             892717 non-null  object 
 10  Situação Voo             917919 non-null  object 
 11  Código Justificativa     0 non-null       float64
dtypes: float64(1), object(11)
memory usage: 84.0+ MB


,ICAO Empresa Aérea,Número Voo,Código Autorização (DI),Código Tipo Linha,ICAO Aeródromo Origem,ICAO Aeródromo Destino,Partida Prevista,Partida Real,Chegada Prevista,Chegada Real,Situação Voo,Código Justificativa
0,GLO,1137,0,N,SBCT,SBSP,2025-09-30 21:00:00,2025-09-30 21:08:00,2025-09-30 22:10:00,2025-09-30 22:30:00,REALIZADO,NaN
1,GLO,1138,0,N,SBFN,SBRF,2025-09-01 13:25:00,2025-09-01 13:18:00,2025-09-01 14:35:00,2025-09-01 14:29:00,REALIZADO,NaN
2,GLO,1138,0,N,SBFN,SBRF,2025-09-03 13:25:00,2025-09-03 13:19:00,2025-09-03 14:35:00,2025-09-03 14:29:00,REALIZADO,NaN
3,GLO,1138,0,N,SBFN,SBRF,2025-09-05 13:25:00,2025-09-05 13:13:00,2025-09-05 14:35:00,2025-09-05 14:19:00,REALIZADO,NaN
4,GLO,1138,0,N,SBFN,SBRF,2025-09-06 13:25:00,2025-09-06 13:16:00,2025-09-06 14:35:00,2025-09-06 14:18:00,REALIZADO,NaN


In [ ]:
#Origem ou destino = SBPA (Aeroporto de POA)
df_sbpa = df[
    (df["ICAO Aeródromo Origem"] == "SBPA") |
    (df["ICAO Aeródromo Destino"] == "SBPA")
]

In [ ]:
partidas_sbpa = df[df["ICAO Aeródromo Origem"] == "SBPA"]

destino_sbpa = df[df["ICAO Aeródromo Destino"] == "SBPA"]


In [ ]:
destino_sbpa[["ICAO Aeródromo Origem", "ICAO Aeródromo Destino"]].value_counts().head()


,,count
ICAO Aeródromo Origem,ICAO Aeródromo Destino,
SBSP,SBPA,7423
SBGR,SBPA,6084
SBGL,SBPA,3455
SBKP,SBPA,2307
SBCT,SBPA,1787


In [ ]:
df_sbpa = df[
    (df["ICAO Aeródromo Origem"] == "SBPA") |
    (df["ICAO Aeródromo Destino"] == "SBPA")
].copy()


In [ ]:
import numpy as np

df_sbpa["movimento"] = np.select(
    [
        df_sbpa["ICAO Aeródromo Origem"] == "SBPA",
        df_sbpa["ICAO Aeródromo Destino"] == "SBPA"
    ],
    [
        "partida",
        "chegada"
    ],
    default="outro"
)


In [ ]:
#Convertendo datas para datetime
time_cols = [
    "Partida Prevista",
    "Partida Real",
    "Chegada Prevista",
    "Chegada Real"
]

for c in time_cols:
    df_sbpa[c] = pd.to_datetime(df_sbpa[c], errors="coerce", utc=True)


In [ ]:
df_sbpa["timestamp"] = np.where(
    df_sbpa["movimento"] == "partida",
    df_sbpa["Partida Real"].fillna(df_sbpa["Partida Prevista"]),
    df_sbpa["Chegada Real"].fillna(df_sbpa["Chegada Prevista"])
)

df_sbpa["timestamp"] = pd.to_datetime(df_sbpa["timestamp"], utc=True)


In [ ]:
voos_por_hora = (
    df_sbpa
    .set_index("timestamp")
    .groupby("movimento")
    .resample("3h")   # <- aqui
    .size()
    .rename("n_voos")
    .reset_index()
)


In [ ]:
print(voos_por_hora)

     movimento                 timestamp  n_voos
0      chegada 2025-01-01 06:00:00+00:00       1
1      chegada 2025-01-01 09:00:00+00:00       8
2      chegada 2025-01-01 12:00:00+00:00       6
3      chegada 2025-01-01 15:00:00+00:00       8
4      chegada 2025-01-01 18:00:00+00:00      11
...        ...                       ...     ...
5337   partida 2025-11-30 09:00:00+00:00      16
5338   partida 2025-11-30 12:00:00+00:00      10
5339   partida 2025-11-30 15:00:00+00:00      10
5340   partida 2025-11-30 18:00:00+00:00      18
5341   partida 2025-11-30 21:00:00+00:00       3

[5342 rows x 3 columns]


In [ ]:
df_sbpa.to_csv("df_sbpa.csv")

#Criação de OO para os voos


| Coluna CSV              | Objeto    | Campo                |
| ----------------------- | --------- | -------------------- |
| ICAO Empresa Aérea      | Voo       | empresa_icao         |
| Número Voo              | Voo       | numero_voo           |
| Código Autorização (DI) | Voo       | codigo_autorizacao   |
| Código Tipo Linha       | Voo       | tipo_linha           |
| ICAO Aeródromo Origem   | Voo       | origem               |
| ICAO Aeródromo Destino  | Voo       | destino              |
| movimento               | EventoVoo | movimento            |
| Partida Prevista        | EventoVoo | partida_prevista     |
| Partida Real            | EventoVoo | partida_real         |
| Chegada Prevista        | EventoVoo | chegada_prevista     |
| Chegada Real            | EventoVoo | chegada_real         |
| Situação Voo            | EventoVoo | situacao             |
| Código Justificativa    | EventoVoo | codigo_justificativa |
| timestamp               | EventoVoo | timestamp            |


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Aeroporto:
    icao: str               # "SBPA"


In [ ]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Voo:
    numero_voo: str                 # "GLO1234"
    empresa_icao: str               # "GLO"
    codigo_autorizacao: str         # DI
    tipo_linha: str                 # REG, NÃO REG, CARGA...
    origem: Aeroporto
    destino: Aeroporto


In [ ]:
from enum import Enum

class MovimentoVoo(Enum):
    CHEGADA = "chegada"
    PARTIDA = "partida"


class SituacaoVoo(Enum):
    REALIZADO = "REALIZADO"
    CANCELADO = "CANCELADO"
    ALTERNADO = "ALTERNADO"
    DESCONHECIDO = "DESCONHECIDO"


In [ ]:
from dataclasses import dataclass
from datetime import datetime
from typing import Optional

@dataclass
class EventoVoo:
    voo: Voo

    movimento: MovimentoVoo

    partida_prevista: Optional[datetime]
    partida_real: Optional[datetime]

    chegada_prevista: Optional[datetime]
    chegada_real: Optional[datetime]

    situacao: SituacaoVoo
    codigo_justificativa: Optional[str]

    aeroporto_referencia: Aeroporto   # SBPA
    timestamp: datetime                # timestamp analítico principal


In [ ]:
from datetime import timedelta

@dataclass
class EventoVoo:
    ...
    def atraso_partida(self) -> Optional[timedelta]:
        if self.partida_prevista and self.partida_real:
            return self.partida_real - self.partida_prevista
        return None

    def atraso_chegada(self) -> Optional[timedelta]:
        if self.chegada_prevista and self.chegada_real:
            return self.chegada_real - self.chegada_prevista
        return None

    def houve_atraso(self, limite_minutos: int = 15) -> bool:
        atraso = self.atraso_chegada() or self.atraso_partida()
        return atraso is not None and atraso.total_seconds() > limite_minutos * 60


Entendendo os voos (se eles destoaram muito da média de tempo de deslocamento e do tempo previsto para o tempo efetivo (chegada e/ou partida))

In [ ]:
import pandas as pd
import numpy as np


In [ ]:
def diff_minutos(real: pd.Series, previsto: pd.Series) -> pd.Series:
    """
    Retorna diferença em minutos (real - previsto)
    """
    return (real - previsto).dt.total_seconds() / 60


In [ ]:
def tempo_voo_minutos(df: pd.DataFrame, real=True) -> pd.Series:
    """
    Calcula duração do voo em minutos
    """
    if real:
        return (df["Chegada Real"] - df["Partida Real"]).dt.total_seconds() / 60
    else:
        return (df["Chegada Prevista"] - df["Partida Prevista"]).dt.total_seconds() / 60


In [ ]:
def enrich_diffs(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["diff_partida_min"] = diff_minutos(
        df["Partida Real"], df["Partida Prevista"]
    )

    df["diff_chegada_min"] = diff_minutos(
        df["Chegada Real"], df["Chegada Prevista"]
    )

    df["tempo_voo_previsto_min"] = tempo_voo_minutos(df, real=False)
    df["tempo_voo_real_min"] = tempo_voo_minutos(df, real=True)

    df["delta_tempo_voo_min"] = (
        df["tempo_voo_real_min"] - df["tempo_voo_previsto_min"]
    )

    return df


#Média histórica por combinação origem–destino

In [ ]:
def baseline_por_rota(df: pd.DataFrame) -> pd.DataFrame:
    """
    Calcula estatísticas por (origem, destino)
    """
    group_cols = ["ICAO Aeródromo Origem", "ICAO Aeródromo Destino"]

    baseline = (
        df.groupby(group_cols)
        .agg(
            media_tempo_voo=("tempo_voo_real_min", "mean"),
            std_tempo_voo=("tempo_voo_real_min", "std"),
            media_diff_partida=("diff_partida_min", "mean"),
            std_diff_partida=("diff_partida_min", "std"),
            media_diff_chegada=("diff_chegada_min", "mean"),
            std_diff_chegada=("diff_chegada_min", "std"),
            n_voos=("Número Voo", "count"),
        )
        .reset_index()
    )

    return baseline


#Estimando thresholds automaticamente

In [ ]:
def add_thresholds(baseline: pd.DataFrame, k: float = 2.0) -> pd.DataFrame:
    baseline = baseline.copy()

    baseline["thr_partida_min"] = (
        baseline["media_diff_partida"] + k * baseline["std_diff_partida"]
    )

    baseline["thr_chegada_min"] = (
        baseline["media_diff_chegada"] + k * baseline["std_diff_chegada"]
    )

    return baseline


Enriquecendo o dataset final com flags

In [ ]:
def flag_outliers(df: pd.DataFrame, baseline: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df = df.merge(
        baseline[
            [
                "ICAO Aeródromo Origem",
                "ICAO Aeródromo Destino",
                "thr_partida_min",
                "thr_chegada_min",
            ]
        ],
        on=["ICAO Aeródromo Origem", "ICAO Aeródromo Destino"],
        how="left",
    )

    df["outlier_partida"] = (
        df["diff_partida_min"].abs() > df["thr_partida_min"]
    )

    df["outlier_chegada"] = (
        df["diff_chegada_min"].abs() > df["thr_chegada_min"]
    )

    df["outlier_evento"] = df["outlier_partida"] | df["outlier_chegada"]

    return df


#Pipeline de leitura e enriquecimento

In [ ]:
df_enriched = enrich_diffs(df_sbpa)

baseline = baseline_por_rota(df_enriched)
baseline = add_thresholds(baseline, k=2.0)

df_final = flag_outliers(df_enriched, baseline)


In [ ]:
print(df_final)

      ICAO Empresa Aérea Número Voo Código Autorização (DI) Código Tipo Linha  \
0                    GLO       1222                       0                 N   
1                    GLO       1222                       0                 N   
2                    GLO       1222                       0                 N   
3                    GLO       1222                       0                 N   
4                    GLO       1222                       0                 N   
...                  ...        ...                     ...               ...   
51816                AZU       9801                       6                 N   
51817                TTL       5689                       2                 C   
51818                GLO       9651                       6                 N   
51819                TTL       5688                       2                 C   
51820                ARG       1233                       2                 X   

      ICAO Aeródromo Origem

In [ ]:
df_final.to_csv('df_sbpa_enriched.csv')
shutil.copy('df_sbpa_enriched.csv','/content/drive/MyDrive/DOUTORADO/DATASETS/DATASETS_PRONTOS/MOVIMENTACAO_AEROPORTO_SBPA_2025.csv')

'/content/drive/MyDrive/DOUTORADO/DATASETS/DATASETS_PRONTOS/MOVIMENTACAO_AEROPORTO_SBPA_2025.csv'